In [ ]:
# Libraries:
import os
import requests
import numpy as np
import das4whales as dw
import matplotlib.pyplot as plt

from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable
from src.signal_functions import spectrogram_analysis_nfft
from src.processing import spectral_decomposition, normalize_band, crop_data, normalize_band, get_clim

### Download example data:

In [ ]:
from src.download_url_file import download_url_file

datapath2save = r"./results"
url = r"http://piweb.ooirsn.uw.edu/das/data/Optasense/NorthCable/TransmitFiber/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-03T15_06_51-0700/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-04T020002Z.h5"

filepath = download_url_file(url, datapath2save)

### Metadata data reading:

In [ ]:
metadata = dw.data_handle.get_acquisition_parameters(filepath, interrogator='optasense')
fs = metadata['fs']
dx = metadata['dx']
nx = metadata['nx']
print(metadata)

### Data reading:

In [ ]:
CHANNEL_STRIDE = 1
# CHANNEL_STRIDE = 4
selected_channels_m = [0, nx*dx, CHANNEL_STRIDE*dx]
selected_channels = [int(x // dx) for x in selected_channels_m]

tr, time, dist, fileBeginTimeUTC = dw.data_handle.load_das_data(filepath, selected_channels, metadata)

print(
    f"DAS: {(dist[-1]-dist[0])/1000:.2f} km cable | "
    f"{time[-1]-time[0]:.1f} s duration | "
    f"dx={np.mean(np.diff(dist)):.1f} m | "
    f"dt={np.mean(np.diff(time))*1000:.2f} ms"
)

### Crop data array:

In [ ]:
T_MIN, T_MAX = 0, 40         # [s]
D_MIN, D_MAX = 10e3, 60e3    # [m]

cropper = True
if cropper: 
    tr_c, dist_crop, time_crop = crop_data(tr, dist, time, D_MIN, D_MAX, T_MIN, T_MAX)
    print('Data cropped!')
else:
    tr_c, dist_crop, time_crop = tr.copy(), dist.copy(), time.copy()

### Spectrogram representation:

In [ ]:
dist_sel = 50e3 # [m]
Nch = np.argmin(np.abs(dist_crop - dist_sel))
signal = tr_c[Nch, :]

Nfft = 512
overlap = 0.9
tspect, fspect, psd, info_spect = spectrogram_analysis_nfft(signal, fs, Nfft, overlap, plotter=False)
_,tbin,fbin,fvalid,_ = info_spect
tspect = tspect + time_crop[0]

flim_min, flim_max = 10,30
fig = plt.figure(figsize=(10, 6))
orig_map=plt.colormaps.get_cmap('jet')
PSDmin = np.nanmin(psd[np.where((fspect >= fvalid))[0],:]) #dB
PSDmax = np.nanmax(psd[np.where((fspect >= fvalid))[0],:]) #dB
if max(fspect)<=5e3:
    plt.pcolormesh(tspect, fspect, psd, cmap=orig_map, vmin=PSDmin, vmax=PSDmax)
    # plt.axhline(fvalid,color='black',linestyle='--',linewidth=4)
    plt.ylabel('Frequency [Hz]')
    plt.ylim(flim_min, flim_max)
else:
    plt.pcolormesh(tspect, fspect*1e-3, psd, cmap=orig_map, vmin=PSDmin, vmax=PSDmax)
    # plt.axhline(fvalid*1e-3,color='black',linestyle='--',linewidth=4)
    plt.ylabel('Frequency [kHz]')
    plt.ylim(flim_min*1e-3, flim_max*1e-3)
plt.title(f'Channel {Nch} | Distance = {dist_crop[Nch]/1e3:.1f} km\nNFFT={Nfft}, Overlap={int(overlap*100)}%')
cbar = plt.colorbar()
cbar.set_label('PSD [dB re 1A$^2$/Hz]', rotation=270, verticalalignment='baseline')
plt.xlabel('Time [s]')
plt.tight_layout()
if cropper:
    PNGname_spect = f'{filename.split('.')[0]}_crop_spect.png'
else: 
    PNGname_spect = f'{filename.split(".")[0]}_spect.png'
fig.savefig(os.path.join(os.path.dirname(filepath), PNGname_spect), dpi=150, bbox_inches='tight', pad_inches=0.1)
plt.show()

### Define spectral bands and perform spectral decomposition

In [ ]:
B1 = [16, 20]  # FW-B [Hz]
B2 = [20, 28]  # FW-A [Hz]
B3 = [3, 15]   # [Hz]

Bx = [15, 27]  # For the conventional representation [Hz]

tr_b01_c, tr_b02_c, tr_b03_c, trf_c = spectral_decomposition(tr_c, fs, [B1, B2, B3, Bx])
print('spectral_decomposition done!')

### Bands normalization:

In [ ]:
perc_norm = 90
b = normalize_band(tr_b03_c, perc=perc_norm)   # Blue: B3
g = normalize_band(tr_b02_c, perc=perc_norm)   # Green: B2
r = normalize_band(tr_b01_c, perc=perc_norm)   # Red: B1
bands = {"Red": B1, "Green": B2, "Blue": B3}

rgb = np.stack([r, g, b], axis=-1)
print('Normalize bands done!')

### Comparison of Conventional Representations vs. MSR:

In [ ]:
LABEL_SIZE = 18; TICK_SIZE = 16; LEGEND_SIZE = 15; CBAR_SIZE = 15
vmin, vmax = get_clim(trf_c, perc=perc_norm)

fig, axes = plt.subplots(1, 2, figsize=(16, 8.2), sharey=True)
im = axes[0].imshow(
    trf_c,
    aspect='auto',
    origin='lower',
    cmap='seismic',
    vmin=vmin, vmax=vmax,
    extent=[time_crop[0], time_crop[-1], dist_crop[0] / 1e3, dist_crop[-1] / 1e3]
    )
axes[0].set_xlabel('Time [s]', fontsize=LABEL_SIZE)
axes[0].set_ylabel('Distance [km]', fontsize=LABEL_SIZE)
axes[0].tick_params(axis='both', which='major', labelsize=TICK_SIZE)
divider0 = make_axes_locatable(axes[0])
cax0 = divider0.append_axes("right", size="5%", pad=0.04)
cbar = fig.colorbar(im, cax=cax0)
cbar.set_label(f'Strain ({Bx[0]}–{Bx[1]} Hz)', fontsize=CBAR_SIZE, rotation=270, verticalalignment='baseline')
cbar.ax.tick_params(labelsize=CBAR_SIZE)

# Right
axes[1].imshow(
    rgb,
    aspect='auto',
    origin='lower',
    extent=[time_crop[0], time_crop[-1], dist_crop[0] / 1e3, dist_crop[-1] / 1e3]
    )
axes[1].set_xlabel('Time [s]', fontsize=LABEL_SIZE)
axes[1].tick_params(axis='both', which='major', labelsize=TICK_SIZE)

divider1 = make_axes_locatable(axes[1])
cax1 = divider1.append_axes("right", size="5%", pad=0.04)
cax1.set_visible(False)
legend = [
    Patch(color='red', label=f'Red: {bands["Red"][0]}–{bands["Red"][1]} Hz'),
    Patch(color='green', label=f'Green: {bands["Green"][0]}–{bands["Green"][1]} Hz'),
    Patch(color='blue', label=f'Blue: {bands["Blue"][0]}–{bands["Blue"][1]} Hz'),
]
axes[1].legend(handles=legend, loc='lower right', framealpha=0.7, fontsize=LEGEND_SIZE)
plt.tight_layout()

if cropper:
    PNGname = f'{filename.split('.')[0]}_crop.png'
else: 
    PNGname = f'{filename.split(".")[0]}.png'

fig.savefig(os.path.join(os.path.dirname(filepath), PNGname), dpi=150, bbox_inches='tight', pad_inches=0.1)
plt.show()